In [13]:
import pandas as pd, json, math, re

METRIC_COLS = [
    "faithfulness_a","faithfulness_b","faithfulness_c",
    "logic_a","logic_b","logic_c",
    "speculation_a","speculation_b","speculation_c",
    "stance_cues_a","stance_cues_b","stance_cues_c",
]

def extract_rating(v):
    if v is None or (isinstance(v, float) and math.isnan(v)): return None
    if isinstance(v, (int,float)) and not isinstance(v, bool): return float(v)
    if isinstance(v, list):
        if v:
            first = v[0]
            if isinstance(first, dict) and "rating" in first: return float(first["rating"])
            if isinstance(first, (int,float)): return float(first)
        return None
    if isinstance(v, dict) and "rating" in v:
        return float(v["rating"])
    s = str(v).strip()
    try:
        obj = json.loads(s)
        return extract_rating(obj)
    except Exception:
        pass
    m = re.search(r'"?rating"?\s*[:=]\s*([0-9]+(?:\.[0-9]+)?)', s, flags=re.I)
    if m: return float(m.group(1))
    m2 = re.search(r'([0-9]+(?:\.[0-9]+)?)', s)
    return float(m2.group(1)) if m2 else None

def clean_and_aggregate(infile, outfile=None, sep=",", groupby="task_id"):
    df = pd.read_csv(infile, sep=sep, dtype=str, keep_default_na=False, na_values=[""])
    for col in ["annotation_id","id","annotator"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="ignore")
    for col in METRIC_COLS:
        if col in df.columns:
            df[col] = df[col].apply(extract_rating)
    numeric_cols = [c for c in METRIC_COLS if c in df.columns]
    if groupby not in df.columns:
        raise KeyError(f"Missing group-by column: {groupby}")
    avg = df.groupby(groupby, dropna=False)[numeric_cols].mean().reset_index()

    # Preference summary
    def summarize_prefs(g):
        counts = g["overall_preference"].value_counts(dropna=True)
        out = {
            "pref_count_Model A": int(counts.get("Model A", 0)),
            "pref_count_Model B": int(counts.get("Model B", 0)),
            "pref_count_Model C": int(counts.get("Model C", 0)),
        }
        if counts.empty:
            winner, top = "", 0
        else:
            top = counts.max()
            winners = [k for k,v in counts.items() if v == top]
            winner = winners[0] if len(winners) == 1 else "TIE"
        n = int(counts.sum())
        out.update({
            "final_overall_preference": winner,
            "final_preference_votes": int(top),
            "num_annotators": n,
            "agreement_rate": (top / n) if n else float("nan")
        })
        return pd.Series(out)

    if "overall_preference" in df.columns:
        pref = df.groupby(groupby, dropna=False).apply(summarize_prefs).reset_index()
        out = avg.merge(pref, on=groupby, how="left")
    else:
        out = avg

    # Add one representative belief/domain/timestamps for readability
    for c in ["belief","domain","created_at","updated_at"]:
        if c in df.columns:
            out = out.merge(df.groupby(groupby, dropna=False)[c].first().reset_index(), on=groupby, how="left")

    # Column order
    metric_order = [c for c in METRIC_COLS if c in out.columns]
    pref_order = [c for c in [
        "pref_count_Model A","pref_count_Model B","pref_count_Model C",
        "final_overall_preference","final_preference_votes","num_annotators","agreement_rate"
    ] if c in out.columns]
    readable = [c for c in ["belief","domain","created_at","updated_at"] if c in out.columns]
    cols = [groupby] + readable + metric_order + pref_order
    out = out[cols + [c for c in out.columns if c not in cols]]

    if outfile:
        out.to_csv(outfile, index=False)
    return out


In [ ]:
df = pd.read_csv("../generated_data/prompt_labeling/prompt_labeling_results/prompt_labeling_result.csv")
df["task_id"] = df["id"].astype(int) % 8
df.to_csv("../generated_data/prompt_labeling/prompt_labeling_results/prompt_labeling_result.csv")

In [ ]:
df_filtered = df[df["annotator"] != 1].reset_index(drop=True)
df_filtered.to_csv("../generated_data/prompt_labeling/prompt_labeling_results/prompt_labeling_result_updated.csv")

In [ ]:
df_clean = clean_and_aggregate("../generated_data/prompt_labeling/prompt_labeling_results/prompt_labeling_result_updated.csv", outfile="../generated_data/prompt_labeling/prompt_labeling_results/cleaned_prompt_labeling_result_updated.csv", sep=",")

df_clean

C:\Users\yuyue\AppData\Local\Temp\ipykernel_87872\3853520331.py:36: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], errors="ignore")
C:\Users\yuyue\AppData\Local\Temp\ipykernel_87872\3853520331.py:69: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  pref = df.groupby(groupby, dropna=False).apply(summarize_prefs).reset_index()


,task_id,belief,domain,created_at,updated_at,faithfulness_a,faithfulness_b,faithfulness_c,logic_a,logic_b,...,stance_cues_a,stance_cues_b,stance_cues_c,pref_count_Model A,pref_count_Model B,pref_count_Model C,final_overall_preference,final_preference_votes,num_annotators,agreement_rate
0,0,OPINION ON WHICH SIDE HAS MORE VALID REASONS I...,IsraelHamas,2025-08-23T11:38:27.139616Z,2025-08-23T11:38:27.139616Z,4.0,3.0,4.0,3.0,3.0,...,3.5,3.5,3.5,0,0,1,TIE,1,2,0.5
1,1,OPINION ON WHETHER WHITE PEOPLE BENEFIT FROM S...,Whitebenefit,2025-08-23T11:16:58.490467Z,2025-08-23T11:16:58.490467Z,4.0,4.0,4.0,4.0,4.5,...,4.0,4.5,4.5,0,1,0,TIE,1,2,0.5
2,2,OPINION ON WHICH SIDE HAS MORE VALID REASONS I...,IsraelHamas,2025-08-23T11:21:57.794899Z,2025-08-23T11:21:57.794899Z,3.0,4.0,3.0,4.0,4.0,...,3.5,4.0,4.0,0,1,1,TIE,1,2,0.5
3,3,OPINION ON THE EXTENT TO WHICH THIS INDIVIDUAL...,Violence,2025-08-23T11:25:48.068982Z,2025-08-23T11:25:48.068982Z,2.0,4.0,3.0,4.0,4.5,...,3.5,4.0,3.5,0,2,0,Model B,2,2,1.0
4,4,OPINION ON WHICH SIDE HAS MORE VALID REASONS I...,IsraelHamas,2025-08-23T11:27:49.160609Z,2025-08-23T11:27:49.160609Z,3.0,4.0,4.0,4.5,4.5,...,4.5,4.5,4.5,0,1,0,TIE,1,2,0.5
5,5,OPINION ON WHETHER WHITE PEOPLE BENEFIT FROM S...,Whitebenefit,2025-08-23T11:30:03.289369Z,2025-08-23T11:30:03.289369Z,4.0,2.0,2.0,4.0,1.5,...,4.5,3.0,3.5,2,0,0,Model A,2,2,1.0
6,6,OPINION ON WHICH SIDE HAS MORE VALID REASONS I...,IsraelHamas,2025-08-23T11:33:10.574001Z,2025-08-23T11:33:10.574001Z,4.0,4.0,4.0,4.5,4.5,...,4.0,4.0,4.0,1,0,0,TIE,1,2,0.5
7,7,OPINION ON WHETHER DIVORCE IN THE USA SHOULD B...,Divorce,2025-08-23T11:35:06.158807Z,2025-08-23T11:35:06.158807Z,4.0,2.0,2.0,3.5,3.0,...,2.5,3.5,3.5,1,0,1,TIE,1,2,0.5


In [26]:
def compute_avg_metric(df, prefix):
    cols = [f"{prefix}_a", f"{prefix}_b", f"{prefix}_c"]
    df[f"avg_{prefix}"] = df[cols].mean(axis=1)
    return df

# Compute averages for each metric type
for metric in ["faithfulness", "logic", "speculation", "stance_cues"]:
    df_clean = compute_avg_metric(df_clean, metric)

df_clean

,task_id,belief,domain,created_at,updated_at,faithfulness_a,faithfulness_b,faithfulness_c,logic_a,logic_b,...,pref_count_Model B,pref_count_Model C,final_overall_preference,final_preference_votes,num_annotators,agreement_rate,avg_faithfulness,avg_logic,avg_speculation,avg_stance_cues
0,0,OPINION ON WHICH SIDE HAS MORE VALID REASONS I...,IsraelHamas,2025-08-23T11:38:27.139616Z,2025-08-23T11:38:27.139616Z,4.0,3.0,4.0,3.0,3.0,...,0,1,TIE,1,2,0.5,3.666667,3.166667,4.000000,3.500000
1,1,OPINION ON WHETHER WHITE PEOPLE BENEFIT FROM S...,Whitebenefit,2025-08-23T11:16:58.490467Z,2025-08-23T11:16:58.490467Z,4.0,4.0,4.0,4.0,4.5,...,1,0,TIE,1,2,0.5,4.000000,4.333333,4.000000,4.333333
2,2,OPINION ON WHICH SIDE HAS MORE VALID REASONS I...,IsraelHamas,2025-08-23T11:21:57.794899Z,2025-08-23T11:21:57.794899Z,3.0,4.0,3.0,4.0,4.0,...,1,1,TIE,1,2,0.5,3.333333,4.000000,3.833333,3.833333
3,3,OPINION ON THE EXTENT TO WHICH THIS INDIVIDUAL...,Violence,2025-08-23T11:25:48.068982Z,2025-08-23T11:25:48.068982Z,2.0,4.0,3.0,4.0,4.5,...,2,0,Model B,2,2,1.0,3.000000,4.166667,4.000000,3.666667
4,4,OPINION ON WHICH SIDE HAS MORE VALID REASONS I...,IsraelHamas,2025-08-23T11:27:49.160609Z,2025-08-23T11:27:49.160609Z,3.0,4.0,4.0,4.5,4.5,...,1,0,TIE,1,2,0.5,3.666667,4.500000,4.166667,4.500000
5,5,OPINION ON WHETHER WHITE PEOPLE BENEFIT FROM S...,Whitebenefit,2025-08-23T11:30:03.289369Z,2025-08-23T11:30:03.289369Z,4.0,2.0,2.0,4.0,1.5,...,0,0,Model A,2,2,1.0,2.666667,2.666667,2.500000,3.666667
6,6,OPINION ON WHICH SIDE HAS MORE VALID REASONS I...,IsraelHamas,2025-08-23T11:33:10.574001Z,2025-08-23T11:33:10.574001Z,4.0,4.0,4.0,4.5,4.5,...,0,0,TIE,1,2,0.5,4.000000,4.500000,3.333333,4.000000
7,7,OPINION ON WHETHER DIVORCE IN THE USA SHOULD B...,Divorce,2025-08-23T11:35:06.158807Z,2025-08-23T11:35:06.158807Z,4.0,2.0,2.0,3.5,3.0,...,0,1,TIE,1,2,0.5,2.666667,3.333333,4.500000,3.166667


In [ ]:
import pandas as pd

df_tasks = df_clean

# Metric columns per model
metric_cols = [
    "faithfulness_a","faithfulness_b","faithfulness_c",
    "logic_a","logic_b","logic_c",
    "speculation_a","speculation_b","speculation_c",
    "stance_cues_a","stance_cues_b","stance_cues_c"
]

# Average numeric metrics across all tasks
avg_metrics = df_tasks[metric_cols].mean().round(3)

# Sum preference counts across all tasks
pref_cols = ["pref_count_Model A","pref_count_Model B","pref_count_Model C"]
total_prefs = df_tasks[pref_cols].sum()

# Overall winner based on total preference counts
overall_winner = total_prefs.idxmax().replace("pref_count_","")

# Agreement rate across all tasks
overall_agreement = df_tasks["agreement_rate"].mean().round(3)

# Combine into one summary dataframe
summary_dict = avg_metrics.to_dict()
summary_dict.update(total_prefs.to_dict())
summary_dict.update({
    "final_overall_preference": overall_winner,
    "overall_agreement_rate": overall_agreement
})

df_overall_summary = pd.DataFrame([summary_dict])

df_overall_summary


,faithfulness_a,faithfulness_b,faithfulness_c,logic_a,logic_b,logic_c,speculation_a,speculation_b,speculation_c,stance_cues_a,stance_cues_b,stance_cues_c,pref_count_Model A,pref_count_Model B,pref_count_Model C,final_overall_preference,overall_agreement_rate
0,3.5,3.375,3.25,3.938,3.688,3.875,4.062,3.75,3.562,3.75,3.875,3.875,4,5,3,Model B,0.625


In [33]:
df_t = df_overall_summary.T
df_t

,0
faithfulness_a,3.5
faithfulness_b,3.375
faithfulness_c,3.25
logic_a,3.938
logic_b,3.688
logic_c,3.875
speculation_a,4.062
speculation_b,3.75
speculation_c,3.562
stance_cues_a,3.75


In [ ]:
import pandas as pd

df_tasks = df_clean
# Metric columns per model
metric_cols = {
    "Model A": ["faithfulness_a","logic_a","speculation_a","stance_cues_a"],
    "Model B": ["faithfulness_b","logic_b","speculation_b","stance_cues_b"],
    "Model C": ["faithfulness_c","logic_c","speculation_c","stance_cues_c"]
}

# Average numeric metrics across all tasks
avg_metrics = {}
overall_score = {}
for model, cols in metric_cols.items():
    avg_val = df_tasks[cols].mean().round(3)
    avg_metrics.update(avg_val.to_dict())
    # Overall score per model: mean of all metrics
    overall_score[model + "_overall_score"] = avg_val.mean().round(3)

# Sum preference counts across all tasks
pref_cols = ["pref_count_Model A","pref_count_Model B","pref_count_Model C"]
total_prefs = df_tasks[pref_cols].sum()

# Overall winner based on total preference counts
overall_winner = total_prefs.idxmax().replace("pref_count_","")  

# Agreement rate across all tasks
overall_agreement = df_tasks["agreement_rate"].mean().round(3)

# Combine into one summary dataframe
summary_dict = {}
summary_dict.update(avg_metrics)
summary_dict.update(overall_score)
summary_dict.update(total_prefs.to_dict())
summary_dict.update({
    "final_overall_preference": overall_winner,
    "overall_agreement_rate": overall_agreement
})

df_overall_summary = pd.DataFrame([summary_dict])

df_overall_summary


,faithfulness_a,logic_a,speculation_a,stance_cues_a,faithfulness_b,logic_b,speculation_b,stance_cues_b,faithfulness_c,logic_c,speculation_c,stance_cues_c,Model A_overall_score,Model B_overall_score,Model C_overall_score,pref_count_Model A,pref_count_Model B,pref_count_Model C,final_overall_preference,overall_agreement_rate
0,3.5,3.938,4.062,3.75,3.375,3.688,3.75,3.875,3.25,3.875,3.562,3.875,3.812,3.672,3.64,4,5,3,Model B,0.625
